In [2]:
from __future__ import annotations

import os

CPU_THREADS = 10
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["KMP_BLOCKTIME"] = "1"
os.environ["KMP_AFFINITY"] = "granularity=fine,compact,1,0"

import time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import torch

from audit_cases import audit_all
from pinn_loader import load_all_cases
from pinn_sampler import SamplerConfig, MultiCaseSampler
from stage5_model import FlowPINNHardBC
from stage5_losses import compute_losses, LossWeights


torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(2)
except RuntimeError:
    pass


@dataclass
class TrainConfig:
    ROOT: Path = Path(r"D:\Projects\Simulations\AI FInal\Ansys\Shapes")
    Re_max: float = 30.0

    width: int = 128
    n_blocks: int = 6
    activation: str = "tanh"

    beta_body: float = 5.0
    beta_wall: float = 5.0
    softplus_k: float = 20.0

    seed: int = 1
    adam_steps: int = 10000
    lr_adam: float = 1e-3
    lr_min: float = 2e-4
    grad_clip: float = 5.0

    warmup_steps: int = 1500
    pde_ramp_steps: int = 5000

    n_data: int = 8192
    n_pde: int = 8192
    n_bc: int = 2048
    n_extra: int = 2048
    mix_mode: str = "balanced"

    dtype: torch.dtype = torch.float64
    log_every: int = 50
    sampler_verbose: bool = False

    save_dir: Path = Path("checkpoints_stage5")
    save_name: str = "stage5_pinn_cpu.pt"

    use_auto_weights: bool = True
    ema_beta: float = 0.99
    w_auto_min: float = 0.1
    w_auto_max: float = 50.0

    use_lbfgs: bool = True
    lbfgs_steps: int = 350
    lbfgs_batches: int = 6

    force_x_bounds: bool = True
    x_min_force: float = 0.0
    x_max_force: float = 22.0


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)


def pick_device_and_dtype(cfg: TrainConfig):
    return torch.device("cpu"), cfg.dtype, "CPU"


def infer_walls_from_cases(cases) -> Tuple[float, float]:
    ys = []
    for c in cases:
        Xw = c.bc["wall"]["X"]
        if Xw is not None and Xw.shape[0] > 0:
            ys.append(Xw[:, 1].astype(np.float64))

    if not ys:
        raise RuntimeError("No wall BC points found. Cannot infer y_bot/y_top.")

    y_all = np.concatenate(ys, axis=0)
    y_bot = float(np.min(y_all))
    y_top = float(np.max(y_all))

    if not (y_top > y_bot):
        raise RuntimeError(f"Bad inferred walls: y_bot={y_bot}, y_top={y_top}")

    return y_bot, y_top


def infer_x_bounds_from_cases(cases, fallback=(0.0, 22.0)) -> Tuple[float, float]:
    xs = []

    for c in cases:
        Xf = c.field_X
        if Xf is not None and Xf.shape[0] > 0:
            xs.append(Xf[:, 0].astype(np.float64))

        for bc_type in ("inlet", "outlet"):
            Xb = c.bc[bc_type]["X"]
            if Xb is not None and Xb.shape[0] > 0:
                xs.append(Xb[:, 0].astype(np.float64))

    if not xs:
        return fallback

    x_all = np.concatenate(xs, axis=0)
    x_min = float(np.min(x_all))
    x_max = float(np.max(x_all))

    if not (x_max > x_min):
        return fallback

    return x_min, x_max


def to_torch_batch(batch, device, dtype):
    batch.X_data = torch.as_tensor(batch.X_data, device=device, dtype=dtype)
    batch.Y_data = torch.as_tensor(batch.Y_data, device=device, dtype=dtype)
    batch.X_pde = torch.as_tensor(batch.X_pde, device=device, dtype=dtype)

    for k in batch.bc:
        batch.bc[k]["X"] = torch.as_tensor(batch.bc[k]["X"], device=device, dtype=dtype)
        batch.bc[k]["Y"] = torch.as_tensor(batch.bc[k]["Y"], device=device, dtype=dtype)

    if getattr(batch, "extra", None) is not None:
        batch.extra["X"] = torch.as_tensor(batch.extra["X"], device=device, dtype=dtype)
        batch.extra["dpdx"] = torch.as_tensor(batch.extra["dpdx"], device=device, dtype=dtype)
        batch.extra["dpdy"] = torch.as_tensor(batch.extra["dpdy"], device=device, dtype=dtype)

    if getattr(batch, "probes", None) is not None:
        batch.probes["front_X"] = torch.as_tensor(batch.probes["front_X"], device=device, dtype=dtype)
        batch.probes["rear_X"] = torch.as_tensor(batch.probes["rear_X"], device=device, dtype=dtype)

    return batch


def cosine_lr(step: int, total: int, lr0: float, lr_min: float) -> float:
    t = min(max(step / max(total, 1), 0.0), 1.0)
    return lr_min + 0.5 * (lr0 - lr_min) * (1.0 + np.cos(np.pi * t))


def pde_lambda(step: int, warmup: int, ramp: int) -> float:
    if step <= warmup:
        return 0.0
    s = step - warmup
    return float(min(1.0, s / max(ramp, 1)))


def save_checkpoint(path: Path, model: torch.nn.Module, cfg: TrainConfig, best: Dict):
    cfg_dict = {}
    for k, v in cfg.__dict__.items():
        cfg_dict[k] = str(v) if isinstance(v, Path) else v

    torch.save(
        {
            "model": model.state_dict(),
            "cfg": cfg_dict,
            "best": best,
        },
        path,
    )


def cat_batches(batches, device, dtype):
    b0 = batches[0]

    b0.X_data = torch.cat([torch.as_tensor(b.X_data, device=device, dtype=dtype) for b in batches], dim=0)
    b0.Y_data = torch.cat([torch.as_tensor(b.Y_data, device=device, dtype=dtype) for b in batches], dim=0)
    b0.X_pde = torch.cat([torch.as_tensor(b.X_pde, device=device, dtype=dtype) for b in batches], dim=0)
    b0.has_data = any(bool(getattr(b, "has_data", False)) for b in batches)

    for bc_type in b0.bc.keys():
        b0.bc[bc_type]["X"] = torch.cat(
            [torch.as_tensor(b.bc[bc_type]["X"], device=device, dtype=dtype) for b in batches],
            dim=0,
        )
        b0.bc[bc_type]["Y"] = torch.cat(
            [torch.as_tensor(b.bc[bc_type]["Y"], device=device, dtype=dtype) for b in batches],
            dim=0,
        )
        b0.bc[bc_type]["has_Y"] = all(bool(b.bc[bc_type]["has_Y"]) for b in batches)

    extras = [getattr(b, "extra", None) for b in batches if getattr(b, "extra", None) is not None]
    if extras:
        b0.extra = {
            "X": torch.cat([torch.as_tensor(e["X"], device=device, dtype=dtype) for e in extras], dim=0),
            "dpdx": torch.cat([torch.as_tensor(e["dpdx"], device=device, dtype=dtype) for e in extras], dim=0),
            "dpdy": torch.cat([torch.as_tensor(e["dpdy"], device=device, dtype=dtype) for e in extras], dim=0),
        }
    else:
        b0.extra = None

    case_ids = {b.case_id for b in batches}
    shapes = {b.shape for b in batches}
    res = {b.Re for b in batches}

    if len(case_ids) == 1 and len(shapes) == 1 and len(res) == 1:
        b0.case_id = batches[0].case_id
        b0.shape = batches[0].shape
        b0.Re = batches[0].Re
        b0.data_used = batches[0].data_used
        b0.targets = dict(batches[0].targets)
        b0.probes = getattr(batches[0], "probes", None)
    else:
        b0.case_id = "mixed"
        b0.shape = "mixed"
        b0.Re = -1
        b0.data_used = "mixed"
        b0.targets = {
            "Cd": float("nan"),
            "Cl": float("nan"),
            "deltap_nd": float("nan"),
            "La_nd": float("nan"),
        }
        b0.probes = None

    return b0


class EMALossBalancer:
    def __init__(
        self,
        beta: float = 0.99,
        eps: float = 1e-12,
        wmin: float = 0.1,
        wmax: float = 50.0,
    ):
        self.beta = float(beta)
        self.eps = float(eps)
        self.wmin = float(wmin)
        self.wmax = float(wmax)

        self.ema_data = None
        self.ema_pde = None
        self.ema_bc = None

    def _ema(self, cur: float, old):
        return cur if old is None else (self.beta * old + (1.0 - self.beta) * cur)

    def _clamp(self, x: float) -> float:
        return max(self.wmin, min(self.wmax, x))

    def step(self, L_data: torch.Tensor, L_pde: torch.Tensor, L_bc: torch.Tensor, has_data: bool):
        ld = float(L_data.detach().cpu())
        lp = float(L_pde.detach().cpu())
        lb = float(L_bc.detach().cpu())

        if has_data:
            self.ema_data = self._ema(ld, self.ema_data)
        self.ema_pde = self._ema(lp, self.ema_pde)
        self.ema_bc = self._ema(lb, self.ema_bc)

        ems = []
        if has_data and self.ema_data is not None:
            ems.append(self.ema_data)
        if self.ema_pde is not None:
            ems.append(self.ema_pde)
        if self.ema_bc is not None:
            ems.append(self.ema_bc)

        mean_ema = sum(ems) / max(len(ems), 1)

        w_data = 0.0
        if has_data and self.ema_data is not None:
            w_data = self._clamp(mean_ema / (self.ema_data + self.eps))

        w_pde = self._clamp(mean_ema / (self.ema_pde + self.eps)) if self.ema_pde is not None else 1.0
        w_bc = self._clamp(mean_ema / (self.ema_bc + self.eps)) if self.ema_bc is not None else 1.0

        return float(w_data), float(w_pde), float(w_bc)


def build_loss_weights() -> LossWeights:
    return LossWeights(
        w_data_u=1.0,
        w_data_v=1.0,
        w_data_p=2.0,
        w_pde=1.0,
        w_cont=1.0,
        w_mom=1.0,
        w_inlet_u=2.0,
        w_inlet_v=2.0,
        w_inlet_p=1.0,
        w_outlet_p=5.0,
        w_wall_u=1.0,
        w_wall_v=1.0,
        w_body_u=1.0,
        w_body_v=1.0,
        w_body_p=5.0,
        w_probe_dp=10.0,
        w_grad_p=1.0,
        w_pde_near_body=4.0,
        near_body_k=5.0,
        phi_min=0.0,
        zero_mean_data_p=False,
    )


def format_case_key(shape, Re) -> str:
    return f"{shape}_Re{Re}"


def print_case_counts(title: str, counts: Counter):
    print(title)
    if not counts:
        print("  (none)")
        return

    print(f"  {'Case':<24} {'Count':>8}")
    print(f"  {'-' * 24} {'-' * 8}")
    for k in sorted(counts):
        print(f"  {k:<24} {counts[k]:>8d}")


def main(cfg: TrainConfig):
    set_seed(cfg.seed)
    device, dtype, backend = pick_device_and_dtype(cfg)
    torch.set_default_dtype(dtype)

    print(f"[DEVICE] backend={backend} device={device} dtype={dtype}")
    print(f"[CPU] torch threads={torch.get_num_threads()} interop={torch.get_num_interop_threads()}")
    print(f"[TORCH] version={torch.__version__}")

    cfg.save_dir.mkdir(parents=True, exist_ok=True)
    save_path = cfg.save_dir / cfg.save_name

    report = cfg.ROOT / "_audit_report.csv"
    audit_all(cfg.ROOT, report)

    cases = load_all_cases(cfg.ROOT, Re_max=cfg.Re_max, verbose=True)
    if not cases:
        print("[TRAIN] No valid cases found. Exiting.")
        raise SystemExit(0)

    print(f"[TRAIN] n_cases={len(cases)}")
    try:
        print("[TRAIN] first_cases:", [(c.paths.shape, c.paths.Re, c.paths.case_id) for c in cases[:10]])
    except Exception:
        pass

    y_bot, y_top = infer_walls_from_cases(cases)

    if cfg.force_x_bounds:
        x_min = float(cfg.x_min_force)
        x_max = float(cfg.x_max_force)
        print(f"[X] FORCED x_min={x_min:.6f}, x_max={x_max:.6f} (length ~ {x_max - x_min:.6f})")
    else:
        x_min, x_max = infer_x_bounds_from_cases(cases, fallback=(0.0, 22.0))
        print(f"[X] inferred x_min={x_min:.6f}, x_max={x_max:.6f} (length ~ {x_max - x_min:.6f})")

    print(f"[WALLS] y_bot={y_bot:.6f}, y_top={y_top:.6f} (height ~ {y_top - y_bot:.6f})")

    sampler = MultiCaseSampler(
        cases,
        SamplerConfig(
            seed=cfg.seed,
            n_data=cfg.n_data,
            n_pde=cfg.n_pde,
            n_bc_inlet=cfg.n_bc,
            n_bc_outlet=cfg.n_bc,
            n_bc_wall=cfg.n_bc,
            n_bc_body=cfg.n_bc,
            n_extra=cfg.n_extra,
            mix_mode=cfg.mix_mode,
            pde_from="field",
            verbose=cfg.sampler_verbose,
        ),
    )

    model = FlowPINNHardBC(
        y_bot=y_bot,
        y_top=y_top,
        x_min=x_min,
        x_max=x_max,
        width=cfg.width,
        n_blocks=cfg.n_blocks,
        activation=cfg.activation,
        beta_body=cfg.beta_body,
        beta_wall=cfg.beta_wall,
        softplus_k=cfg.softplus_k,
    ).to(device=device, dtype=dtype)

    w = build_loss_weights()

    balancer = EMALossBalancer(
        beta=cfg.ema_beta,
        wmin=cfg.w_auto_min,
        wmax=cfg.w_auto_max,
    )
    last_auto = (1.0, 1.0, 1.0)

    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr_adam)

    best = {"loss": float("inf"), "step": 0}
    t0 = time.time()

    case_counts_total = Counter()
    case_counts_block = Counter()
    last_log_time = t0
    last_log_step = 0

    for step in range(1, cfg.adam_steps + 1):
        iter_t0 = time.time()

        lr = cosine_lr(step, cfg.adam_steps, cfg.lr_adam, cfg.lr_min)
        for g in opt.param_groups:
            g["lr"] = lr

        lam_pde = pde_lambda(step, cfg.warmup_steps, cfg.pde_ramp_steps)

        b = sampler.next_batch()
        case_key = format_case_key(b.shape, b.Re)
        case_counts_total[case_key] += 1
        case_counts_block[case_key] += 1

        b = to_torch_batch(b, device, dtype)

        opt.zero_grad(set_to_none=True)

        parts = compute_losses(
            model,
            b,
            Re_max=cfg.Re_max,
            weights=w,
            outlet_p_target=0.0,
        )

        has_data = bool(getattr(b, "has_data", False))

        if cfg.use_auto_weights:
            w_data_auto, w_pde_auto, w_bc_auto = balancer.step(
                parts["L_data"], parts["L_pde"], parts["L_bc"], has_data=has_data
            )
        else:
            w_data_auto, w_pde_auto, w_bc_auto = (1.0, 1.0, 1.0)

        last_auto = (w_data_auto, w_pde_auto, w_bc_auto)

        L_total = (
            (w_data_auto * parts["L_data"]) +
            (w_pde_auto * w.w_pde * lam_pde * parts["L_pde"]) +
            (w_bc_auto * parts["L_bc"]) +
            (w.w_probe_dp * parts["L_probe_dp"]) +
            (w.w_grad_p * parts["L_grad_p"])
        )

        if not torch.isfinite(L_total):
            print(f"[WARN] Non-finite loss at step={step}. Skipping step.")
            continue

        L_total.backward()

        if cfg.grad_clip and cfg.grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

        opt.step()

        L_item = float(L_total.detach().cpu())

        if L_item < best["loss"]:
            best["loss"] = L_item
            best["step"] = step
            save_checkpoint(save_path, model, cfg, best)

        if step == 1 or step % cfg.log_every == 0:
            now = time.time()
            elapsed = now - t0
            step_time = now - iter_t0

            block_steps = step - last_log_step
            block_dt = now - last_log_time
            avg_step_time = block_dt / max(block_steps, 1)

            eta_sec = avg_step_time * (cfg.adam_steps - step)
            eta_min = eta_sec / 60.0

            print(
                f"[Adam {step:6d}/{cfg.adam_steps}] "
                f"L={L_item:.3e} | "
                f"lam_pde={lam_pde:.2f} lr={lr:.2e} | "
                f"step_time={step_time:.2f}s avg_step={avg_step_time:.2f}s elapsed={elapsed/60.0:.1f} min eta={eta_min:.1f} min | "
                f"auto=[d:{w_data_auto:.2f}, p:{w_pde_auto:.2f}, bc:{w_bc_auto:.2f}] | "
                f"data={parts['L_data'].item():.3e} "
                f"(u={parts['L_data_u'].item():.3e}, v={parts['L_data_v'].item():.3e}, p={parts['L_data_p'].item():.3e}) | "
                f"pde={parts['L_pde'].item():.3e} "
                f"(cont={parts['L_cont'].item():.3e}, mom={parts['L_mom'].item():.3e}) | "
                f"bc={parts['L_bc'].item():.3e} "
                f"(in_u={parts['L_inlet_u'].item():.3e}, in_v={parts['L_inlet_v'].item():.3e}, "
                f"in_p={parts['L_inlet_p'].item():.3e}, out_p={parts['L_outlet_p'].item():.3e}, "
                f"body_p={parts['L_body_p'].item():.3e}) | "
                f"probe_dp={parts['L_probe_dp'].item():.3e} "
                f"(pred_dp={parts['pred_deltap_nd'].item():.3e}) | "
                f"grad_p={parts['L_grad_p'].item():.3e} "
                f"(px={parts['L_grad_px'].item():.3e}, py={parts['L_grad_py'].item():.3e}) | "
                f"case={b.shape} Re={b.Re} id={b.case_id} has_data={has_data} | "
                f"data_used={getattr(b, 'data_used', 'na')} | "
                f"target_dp={float(getattr(b, 'targets', {}).get('deltap_nd', float('nan'))):.3g} | "
                f"extra={0 if getattr(b, 'extra', None) is None else b.extra['X'].shape[0]} | "
                f"probes={'yes' if getattr(b, 'probes', None) is not None else 'no'} | "
                f"best@{best['step']}={best['loss']:.3e}"
            )

            print_case_counts("[Case count in last log block]", case_counts_block)
            case_counts_block.clear()
            last_log_time = now
            last_log_step = step

    print(f"[SAVE] best checkpoint: {save_path}")
    print_case_counts("[Adam total case usage]", case_counts_total)

    if cfg.use_lbfgs:
        print("[TRAIN] L-BFGS: building larger fixed batch ...")
        batches = [sampler.next_batch() for _ in range(cfg.lbfgs_batches)]
        b0 = cat_batches(batches, device, dtype)

        has_data0 = bool(getattr(b0, "has_data", False))
        w_data0, w_pde0, w_bc0 = last_auto
        if not has_data0:
            w_data0 = 0.0

        opt2 = torch.optim.LBFGS(
            model.parameters(),
            lr=1.0,
            max_iter=cfg.lbfgs_steps,
            history_size=50,
            line_search_fn="strong_wolfe",
        )

        def closure():
            opt2.zero_grad(set_to_none=True)

            parts2 = compute_losses(
                model,
                b0,
                Re_max=cfg.Re_max,
                weights=w,
                outlet_p_target=0.0,
            )

            L = (
                (w_data0 * parts2["L_data"]) +
                (w_pde0 * w.w_pde * parts2["L_pde"]) +
                (w_bc0 * parts2["L_bc"]) +
                (w.w_probe_dp * parts2["L_probe_dp"]) +
                (w.w_grad_p * parts2["L_grad_p"])
            )

            if not torch.isfinite(L):
                return torch.zeros((), device=device, dtype=dtype)

            L.backward()
            return L

        lbfgs_t0 = time.time()
        print(f"[TRAIN] L-BFGS start ... max_iter={cfg.lbfgs_steps}")
        opt2.step(closure)
        lbfgs_dt = time.time() - lbfgs_t0

        final_parts = compute_losses(
            model,
            b0,
            Re_max=cfg.Re_max,
            weights=w,
            outlet_p_target=0.0,
        )
        L_final = (
            (w_data0 * final_parts["L_data"]) +
            (w_pde0 * w.w_pde * final_parts["L_pde"]) +
            (w_bc0 * final_parts["L_bc"]) +
            (w.w_probe_dp * final_parts["L_probe_dp"]) +
            (w.w_grad_p * final_parts["L_grad_p"])
        )
        L_final_item = float(L_final.detach().cpu())

        print(
            f"[L-BFGS done] "
            f"time={lbfgs_dt:.1f}s | "
            f"L={L_final_item:.3e} | "
            f"data={final_parts['L_data'].item():.3e} "
            f"pde={final_parts['L_pde'].item():.3e} "
            f"bc={final_parts['L_bc'].item():.3e} "
            f"probe_dp={final_parts['L_probe_dp'].item():.3e} "
            f"grad_p={final_parts['L_grad_p'].item():.3e}"
        )

        if L_final_item < best["loss"]:
            best["loss"] = L_final_item
            best["step"] = cfg.adam_steps + cfg.lbfgs_steps

        save_checkpoint(save_path, model, cfg, best)
        print(f"[SAVE] after L-BFGS: {save_path}")

    print(f"[TRAIN] done. best_step={best['step']} best_loss={best['loss']:.3e}")


if __name__ == "__main__":
    main(TrainConfig())

[DEVICE] backend=CPU device=cpu dtype=torch.float64
[CPU] torch threads=10 interop=2
[TORCH] version=2.10.0+cpu

=== AUDIT SUMMARY ===
root: D:\Projects\Simulations\AI FInal\Ansys\Shapes
PASS: 9 | FAIL: 0
report: D:\Projects\Simulations\AI FInal\Ansys\Shapes\_audit_report.csv

=== CASES USED FOR TRAINING (PASS) ===
- Circle | Re=10 | case_id=Circle_2D_Re10 | field_core_rows=153716 | data=pde_field | supervised_uvp | bc:inlet,outlet,wall,body | targets:Cd,Cl,DeltaP,La
- Circle | Re=20 | case_id=Circle_2D_Re20 | field_core_rows=153716 | data=pde_field | supervised_uvp | bc:inlet,outlet,wall,body | targets:Cd,Cl,DeltaP,La
- Circle | Re=30 | case_id=Circle_2D_Re30 | field_core_rows=153716 | data=pde_field | supervised_uvp | bc:inlet,outlet,wall,body | targets:Cd,Cl,DeltaP,La
- Hexagon | Re=10 | case_id=Hexagon_2D_Re10 | field_core_rows=184614 | data=pde_field | supervised_uvp | bc:inlet,outlet,wall,body | targets:Cd,Cl,DeltaP,La
- Hexagon | Re=20 | case_id=Hexagon_2D_Re20 | field_core_rows